# 06. Image Content Moderation via Agent Content Filters

This notebook extends notebook 05's automatic content filtering to a **multimodal** request —
text plus an image (`support.png`, a benign VPN error screenshot) — sent through the same
Foundry connected agent, again inspecting `response.model_extra["content_filters"]`. It shows
that Azure's built-in content filtering scans image inputs, not just text.

**Difficulty:** Intermediate — combines the multimodal input pattern from notebook 01 with the
content-filter inspection pattern from notebook 05.

## Prerequisites

**Python packages** (already in the repo's root `requirements.txt` — no extra install needed):
- `azure-ai-projects`, `azure-identity`, `python-dotenv`
- `base64` — standard library

**Azure resources**
- The same Azure AI Foundry **project** and **connected agent** as notebook 05.

**Environment variables** — reuse the same names as notebook 05, added to the root `.env`:
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_AGENT_NAME=cloudxeus-support
```

**Input file:** `support.png` must exist in the same folder as this notebook (it already does,
as a sample asset).

## What You'll Learn

- That Azure OpenAI/Foundry's built-in content filtering applies to **image inputs** as well as
  text — both are scanned before/alongside generating a response
- How to combine the multimodal `input_text`/`input_image` content-part pattern (from
  notebook 01) with the `agent_reference` + content-filter-inspection pattern (from notebook 05)
- Why this "automatic, embedded" filtering approach differs from explicitly calling Content
  Safety's `analyze_image()` (notebook 07), which gives direct per-category severity scores
  rather than a pass/flag decision bundled with a generation

### Imports and setup

Same as notebook 05, plus `base64` (needed here because this request includes an image, unlike
notebook 05's text-only prompt).

In [ ]:
import os
import base64

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

### Building the project client

Identical to notebook 05 — same project endpoint, same agent, same client construction.

💡 **Exam tip:** Multimodal content filtering coverage (image categories analyzed alongside
text) is a capability specific to multimodal-capable deployments — a text-only model deployment
has nothing to apply image content filtering to in the first place, since it can't accept image
input at all.

In [ ]:
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "cloudxeus-support")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

### Reading and encoding the image

Same base64-encoding pattern as notebook 01: read `support.png` in binary mode and encode it as
a base64 string for embedding in a `data:` URI.

In [ ]:
with open("support.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

### Sending the multimodal request through the agent

This combines the multimodal content-part structure from notebook 01 (`input_text` +
`input_image` inside a `content` list) with the `agent_reference` routing from notebook 05. The
image here is a benign screenshot of a VPN connection error — expected to pass content filtering
cleanly, in contrast to notebook 05's deliberately unsafe text prompt.

🔄 **Alternatives:** For a case where you specifically want to test whether an *image* (rather
than the accompanying text) trips content filtering, you'd swap in an image containing content
from one of the built-in categories (violence, sexual content, etc.) instead of this benign
screenshot.

In [ ]:
response = openai.responses.create(
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "I'm getting this error connecting to the VPN. What does it mean?"},
                {"type": "input_image", "image_url": f"data:image/png;base64,{image_b64}"},
            ],
        }
    ],
)

### Inspecting the response and its content filter results

Same pattern as notebook 05: print the generated text answer, then print the content filter
metadata. Since `support.png` is a benign screenshot, expect low/safe severity across categories
here — a useful contrast against notebook 05's flagged result.

In [ ]:
print(response.output_text)
print(response.model_extra["content_filters"])

## Summary

This notebook sent a multimodal (text + image) request through the same Foundry connected agent
used in notebook 05, and confirmed that Azure's automatic content filtering covers image inputs
too — the `content_filters` metadata is present on multimodal responses the same way it is on
text-only ones. Because `support.png` is a benign screenshot, this run should show clean/low
severity results, contrasting with notebook 05's deliberately flagged text prompt.

## Try It Yourself

1. **Easy:** Swap `support.png` for `sales_data.png` or `discover.png` (also in this folder) and
   compare the `content_filters` output across different benign images.
2. **Intermediate:** Ask a different, more open-ended question about the same image (e.g. "What
   troubleshooting steps would you suggest?") and see whether the content filter results change
   based on prompt wording alone, holding the image constant.
3. **Advanced:** Compare this notebook's `content_filters` output structure against notebook 07's
   `categories_analysis` output structure from the standalone Content Safety service on the same
   image — what information does each approach give you that the other doesn't?